# Train the two policy models (GPU)

Model 1 **TilePolicyNet**: state -> per-tile edge prediction (masked over
the graph's own legal edges). Model 2 **MarketPolicyNet**: state + chosen
edges -> the day's 24x10 market slot program (kind/item/qty heads).

Data: the `kaggriculture-edge-dataset` shards (440 episodes). Exports both
models as pure-NumPy npz, verified against JAX before writing.

In [ ]:
# 0. environment: locate the code tree (agent/) ANYWHERE under /kaggle/input
import os, shutil, glob, subprocess
WORK = "/kaggle/working"
_cands = glob.glob("/kaggle/input/**/agent/artifact", recursive=True)
print("input scan:", glob.glob("/kaggle/input/*"), glob.glob("/kaggle/input/*/*")[:6])
assert _cands, "agent/artifact not found anywhere under /kaggle/input"
CODE = os.path.dirname(os.path.dirname(_cands[0]))   # dir that CONTAINS agent/
shutil.copytree(CODE, WORK, dirs_exist_ok=True)
os.chdir(WORK)
subprocess.run(["pip", "install", "-q", "kaggle-environments==1.32.7"], check=True)
print("code ready:", os.path.isdir("agent/artifact"), "from", CODE)

In [ ]:
# 1. corpus table — self-contained (no offline_lab imports)
import sys, glob, json, os
import numpy as np
sys.path.insert(0, WORK)

QTY_BUCKETS = [0, 1, 2, 4, 8, 16, 10**9]

def _scalars_of(sh):
    n = int(sh["n_samples"])
    scal = np.zeros((n, 40), dtype=np.float32)
    scal[:, 0] = sh["money_own"]
    scal[:, 1] = sh["money_opp"]
    scal[:, 2:14] = sh["shed_own"]
    scal[:, 14:26] = sh["shed_opp"]
    scal[:, 26:31] = sh["seeds"]
    scal[:, 31:40] = sh["prices"]
    scal[:, 39] = sh["day"] / 29.0
    return scal

def _edge_summary(sh):
    n = int(sh["n_samples"])
    out = np.zeros((n, N_CHAINS_CAP), dtype=np.float32)
    for i in range(n):
        lo, hi = int(sh["lab_ptr"][i]), int(sh["lab_ptr"][i+1])
        for j in range(lo, hi):
            cid = int(sh["lab_chain"][j])
            if 0 <= cid < N_CHAINS_CAP:
                out[i, cid] += 1.0
    return out

def _order_targets(sh):
    n = int(sh["n_samples"])
    kind = np.zeros((n, 24, 10), dtype=np.int32)
    item = -np.ones((n, 24, 10), dtype=np.int32)
    qty = np.zeros((n, 24, 10), dtype=np.int32)
    for i in range(n):
        for j in range(int(sh["ord_ptr"][i]), int(sh["ord_ptr"][i+1])):
            t, s = int(sh["ord_hour"][j]), int(sh["ord_idx"][j])
            if t < 24 and s < 10:
                kind[i, t, s] = 1 + int(sh["ord_code"][j])
                item[i, t, s] = int(sh["ord_item"][j])
                q = int(sh["ord_qty"][j])
                qty[i, t, s] = next(b for b, cap in enumerate(QTY_BUCKETS) if q <= cap)
    return kind, item, qty

def load_corpus(paths):
    tiles_own, tiles_opp, scal, chain_bag = [], [], [], []
    lab_ptr, lab_cell, lab_chain, lab_item = [0], [], [], []
    kind_g, item_g, qty_g = [], [], []
    for p in paths:
        sh = np.load(p, allow_pickle=False)
        tiles_own.append(sh["tiles_own"])
        tiles_opp.append(sh["tiles_opp"])
        scal.append(_scalars_of(sh))
        chain_bag.append(_edge_summary(sh))
        lab_cell.append(sh["lab_cell"])
        lab_chain.append(sh["lab_chain"])
        lab_item.append(sh["lab_item"])
        counts = np.diff(sh["lab_ptr"])
        prev = lab_ptr[-1]
        for c in counts:
            prev += int(c)
            lab_ptr.append(prev)
        kg, ig, qg = _order_targets(sh)
        kind_g.append(kg); item_g.append(ig); qty_g.append(qg)
    return {
        "tiles_own": np.concatenate(tiles_own),
        "tiles_opp": np.concatenate(tiles_opp),
        "scalars": np.concatenate(scal),
        "chain_bag": np.concatenate(chain_bag),
        "lab_cell": np.concatenate(lab_cell),
        "lab_chain": np.concatenate(lab_chain),
        "lab_item": np.concatenate(lab_item),
        "lab_ptr": np.array(lab_ptr, dtype=np.int64),
        "ord_kind": np.concatenate(kind_g),
        "ord_item": np.concatenate(item_g),
        "ord_qty": np.concatenate(qty_g),
    }

N_CHAINS_CAP = 64
paths = sorted(p for p in glob.glob("/kaggle/input/**/*.npz", recursive=True)
               if os.path.basename(p)[:1].isdigit() and "/input/" in p)
assert paths, "dataset shards not attached"
# tiles arrays hold PACKED state keys; the embedding needs dense row ids.
# Build the map once from the graph (LOCKED -1 -> row 703), remap all shards.
from agent.planner.inputs import GRAPH_PATH as _GP
from agent.tile_dp.graph import TileGraph
graph = TileGraph.load(_GP)
_key_index = graph.key_index
_lock_row = 703
key2row = {}
for k in set(int(x) for p in paths for x in np.load(p)["tiles_own"].ravel()):
    if k < 0:
        key2row[k] = _lock_row
    else:
        key2row[k] = _key_index.get(k, _lock_row)

def _remap(sh):
    v = sh["tiles_own"].copy()
    for kk, row in key2row.items():
        v[v == kk] = row
    return v

table = load_corpus(paths)
for b in range(len(paths)):
    sh = np.load(paths[b], allow_pickle=False)
    pass
# remap tiles_own sample-by-sample (order matches concatenation)
rows = []
for p in paths:
    sh = np.load(p, allow_pickle=False)
    rows.append(_remap(sh))
table["tiles_own"] = np.concatenate(rows)
n = table["tiles_own"].shape[0]
print(f"corpus: {n} samples from {len(paths)} shards; tile rows remapped")

In [ ]:
# 2. Model 1 — TilePolicyNet (masked edge softmax per tile)
import jax, jax.numpy as jnp, optax
from agent.planner.inputs import GRAPH_PATH as _GP
from agent.tile_dp.graph import TileGraph
from agent.tile_dp.chains import loaded_chains
graph = TileGraph.load(_GP)
N_CHAINS = len(loaded_chains())
D = 32
k = jax.random.PRNGKey(7)
k1,k2,k3,k4,k5,k6 = jax.random.split(k, 6)
def glorot(shape, kk):
    return jax.random.normal(kk, shape) * np.sqrt(2.0/(shape[0]+shape[1]))
params = {
    "tile_emb": glorot((704, D), k1),
    "edge_emb": glorot((N_CHAINS, D), k2),
    "w_sc": glorot((40, D), k3), "b_sc": jnp.zeros((D,)),
    "w_t": glorot((2*D, D), k4), "b_t": jnp.zeros((D,)),
}
def loss_m1(params, tiles_own, scal, lab_chain, edge_lo, edge_hi, edge_chain):
    """Mean CE over labeled tiles; candidates = the state's own edges."""
    t = params["tile_emb"][tiles_own]                    # (B,100,d)
    ctx = jnp.tanh(scal @ params["w_sc"] + params["b_sc"])
    ctx = jnp.repeat(ctx[:, None, :], 100, axis=1)
    tv = jnp.tanh(jnp.concatenate([t, ctx], -1) @ params["w_t"] + params["b_t"])
    ee = params["edge_emb"][edge_chain]                  # (B,100,E,d)
    score = jnp.einsum("bkd,bked->bke", tv, ee)          # (B,100,E)
    # label position: which candidate row matches the label chain
    match = (edge_chain == lab_chain[:, :, None])        # (B,100,E)
    any_match = match.any(-1)
    pos = jnp.argmax(match.astype(jnp.int32), -1)        # first matching row
    logp = jax.nn.log_softmax(score, -1)
    picked = jnp.take_along_axis(logp, pos[..., None], -1)[..., 0]
    ce = -jnp.where(any_match, picked, 0.0)
    denom = any_match.sum()
    return jnp.where(denom > 0, ce.sum() / jnp.maximum(denom, 1), 0.0)
print("model 1 defined")

In [ ]:
# 3. Model 2 — MarketPolicyNet (24x10 slot program)
NV = 40  # vocab bracket
print("model 2 defined")

@jax.jit
def jax_jit_hidden(p, tiles, scal, bag):
    t = p["tile_emb"][tiles]
    pool = t.mean(axis=1)
    cb = p["chain_emb"][jnp.clip(bag.argmax(-1), 0, p["chain_emb"].shape[0] - 1)]
    ctx = jnp.tanh(scal @ p["w_sc"] + p["b_sc"])
    h = jnp.concatenate([pool, cb, ctx], -1)
    h = jnp.tanh(h @ p["w_h1"] + p["b_h1"])
    h = jnp.tanh(h @ p["w_h2"] + p["b_h2"])
    return h


In [ ]:
# 4. batch assembly: candidate edges per tile from the graph's CSR (Model 1)
E = 12  # max edges per state window (graph max is under this)
key_index = graph.key_index
state_keys = np.asarray(graph.state_keys)
sid_of = {int(k): i for i, k in enumerate(state_keys)}
edge_lo = np.zeros((n, 100), dtype=np.int32)
edge_hi = np.zeros((n, 100), dtype=np.int32)
edge_chain = np.zeros((n, 100, E), dtype=np.int32)
for b in range(n):
    for c in range(100):
        sk = int(table["tiles_own"][b, c])
        sid = key_index.get(sk, -1)
        if sid < 0:
            sid = 0
        lo, hi = graph.edges_of(sid)
        edge_lo[b, c], edge_hi[b, c] = lo, min(lo + E, hi)
        m = min(lo + E, hi) - lo
        edge_chain[b, c, :m] = graph.edge_chain[lo:lo+m]
print("edge windows assembled")

In [ ]:
# 5. training loop — stabilized recipe (warmup+cosine, clip, balanced heads)
import time, json, shutil
import optax

# class weights from effective counts (Cui et al. 2019): w_c = (1-b)/(1-b^n_c)
def class_weights(counts, b=0.999):
    counts = np.asarray(counts, dtype=np.float64)
    w = (1.0 - b) / (1.0 - np.power(b, np.maximum(counts, 1.0)))
    w = w / w.mean()
    return jnp.array(w.astype(np.float32))

# split by EPISODE (held-out episodes measure generalization)
ep_per_sample = []
for _p in paths:
    _sh = np.load(_p, allow_pickle=False)
    ep_per_sample += [int(_sh["episode_id"][0])] * int(_sh["n_samples"])
table["episode_id"] = np.array(ep_per_sample)
_eps = sorted(set(ep_per_sample))
_rng = np.random.RandomState(7)
_rng.shuffle(_eps)
val_eps = set(_eps[:45])
tr_idx = [i for i in range(n) if table["episode_id"][i] not in val_eps]
va_idx = [i for i in range(n) if table["episode_id"][i] in val_eps]
print(f"split: {len(tr_idx)} train / {len(va_idx)} val "
      f"({len(_eps)-len(val_eps)} / {len(val_eps)} episodes)", flush=True)

# chain-class weights for model 1, from the train split's labels
tr_counts = np.zeros(N_CHAINS, dtype=np.int64)
for i in tr_idx:
    lo, hi = table["lab_ptr"][i], table["lab_ptr"][i+1]
    for c in table["lab_chain"][lo:hi]:
        if c >= 0:
            tr_counts[c] += 1
w1 = class_weights(tr_counts)

# order-kind weights for model 2, from the train split's slots
NV = int(max(table['ord_kind'].max(), table['ord_item'].max() + 1)) + 1  # real vocab bracket
k_counts = np.zeros(NV + 1, dtype=np.int64)
for i in tr_idx:
    for t in range(24):
        for s in range(10):
            k_counts[table["ord_kind"][i, t, s]] += 1
w2k = class_weights(k_counts)
w2k = class_weights(k_counts)
k = jax.random.PRNGKey(11)
k1,k2,k3,k4,k5,k6,k7 = jax.random.split(k, 7)
p2 = {
    "tile_emb": glorot((704, D), k1),
    "chain_emb": glorot((N_CHAINS, D), k2),
    "w_sc": glorot((40, D), k3), "b_sc": jnp.zeros((D,)),
    "w_h1": glorot((3*D, 128), k4), "b_h1": jnp.zeros((128,)),
    "w_h2": glorot((128, 128), k5), "b_h2": jnp.zeros((128,)),
    "w_kind": glorot((128, 24*10*(NV+1)), k6),
    "w_item": glorot((128, 24*10*(NV+1)), k7),
    "w_qty": glorot((128, 24*10*8), k5),
}
def loss_m2(p, tiles_own, scal, chain_bag, kind, item, qty):
    t = p["tile_emb"][tiles_own]                         # (B,100,d)
    pool = t.mean(axis=1)                                # (B,d)
    cb = p["chain_emb"][jnp.clip(chain_bag.argmax(-1), 0, N_CHAINS-1)]
    ctx = jnp.tanh(scal @ p["w_sc"] + p["b_sc"])
    h = jnp.concatenate([pool, cb, ctx], -1)
    h = jnp.tanh(h @ p["w_h1"] + p["b_h1"])
    h = jnp.tanh(h @ p["w_h2"] + p["b_h2"])
    klog = jax.nn.log_softmax((h @ p["w_kind"]).reshape(-1, 24, 10, NV+1), -1)
    ilog = jax.nn.log_softmax((h @ p["w_item"]).reshape(-1, 24, 10, NV+1), -1)
    qlog = jax.nn.log_softmax((h @ p["w_qty"]).reshape(-1, 24, 10, 8), -1)
    one = lambda logp, tgt: jnp.take_along_axis(logp, tgt[..., None], -1)[..., 0]
    kmask = (kind > 0)
    l_k = -(one(klog, kind) * kmask).sum() / jnp.maximum(kmask.sum(), 1)
    l_i = -(one(ilog, item+1) * kmask).sum() / jnp.maximum(kmask.sum(), 1)
    l_q = -(one(qlog, qty) * kmask).sum() / jnp.maximum(kmask.sum(), 1)
    return l_k + l_i + l_q



schedule = optax.warmup_cosine_decay_schedule(
    init_value=1e-4, peak_value=1e-3,
    warmup_steps=200, decay_steps=10 * ((len(tr_idx) // 128) + 1),
    end_value=1e-5)

opt1 = optax.chain(optax.clip_by_global_norm(1.0), optax.adamw(schedule))
opt2 = optax.chain(optax.clip_by_global_norm(1.0), optax.adamw(schedule))


# ---- checkpointing: weights + optimizer states per epoch; resume on rerun ----
CKPT = f"{WORK}/ckpt"
os.makedirs(CKPT, exist_ok=True)
_p1, _t1 = jax.tree_util.tree_flatten(params)
_p2, _t2 = jax.tree_util.tree_flatten(p2)
_o1, _ot1 = jax.tree_util.tree_flatten(s1)
_o2, _ot2 = jax.tree_util.tree_flatten(s2)

def _save_ckpt(epoch):
    ck = {"epoch": np.int64(epoch)}
    for i, leaf in enumerate(_p1):
        ck["p_" + str(i)] = np.asarray(leaf)
    for i, leaf in enumerate(_p2):
        ck["q_" + str(i)] = np.asarray(leaf)
    for i, leaf in enumerate(_o1):
        ck["o1_" + str(i)] = np.asarray(leaf)
    for i, leaf in enumerate(_o2):
        ck["o2_" + str(i)] = np.asarray(leaf)
    np.savez_compressed(f"{CKPT}/last.npz", **ck)
    # session-independent backup: a dedicated checkpoint DATASET, every 3 epochs
    # (dataset versioning survives any session death; the API call runs inside
    # the kernel with its own credentials)
    if epoch % 3 == 2 or epoch >= 11:
        import subprocess, tempfile
        _up = tempfile.mkdtemp()
        shutil.copy(f"{CKPT}/last.npz", _up + "/last.npz")
        json.dump({"title": "Kaggriculture Policy Train Checkpoint",
                   "id": "amirelite/kaggriculture-policy-train-ckpt",
                   "licenses": [{"name": "CC0-1.0"}]},
                  open(_up + "/dataset-metadata.json", "w"))
        _py = (
            "from kaggle.api.kaggle_api_extended import KaggleApi\n"
            "api = KaggleApi(); api.authenticate()\n"
            "try:\n"
            "    print(api.dataset_create_new(folder=" + repr(_up) + "))\n"
            "except Exception as e:\n"
            "    print('ckpt push:', e)\n")
        open(_up + "/_push.py", "w").write(_py)
        subprocess.run(["python", _up + "/_push.py"], timeout=600)

start_epoch = 0
if os.path.exists(f"{CKPT}/last.npz"):
    _d = np.load(f"{CKPT}/last.npz", allow_pickle=False)
    params = jax.tree_util.tree_unflatten(_t1, [jnp.array(_d["p_" + str(i)])
                                               for i in range(len(_p1))])
    p2 = jax.tree_util.tree_unflatten(_t2, [jnp.array(_d["q_" + str(i)])
                                            for i in range(len(_p2))])
    s1 = jax.tree_util.tree_unflatten(_ot1, [jnp.array(_d["o1_" + str(i)])
                                             for i in range(len(_o1))])
    s2 = jax.tree_util.tree_unflatten(_ot2, [jnp.array(_d["o2_" + str(i)])
                                             for i in range(len(_o2))])
    start_epoch = int(_d["epoch"]) + 1
    print(f"resumed from checkpoint at epoch {start_epoch}", flush=True)


B = 128
STEP = {"i": 0}
t0 = time.time()
history = []
for epoch in range(start_epoch, 12):
    perm = np.random.permutation(len(tr_idx))
    tot1 = tot2 = cnt = skipped = 0
    for b0 in range(0, len(tr_idx), B):
        idx = [tr_idx[i] for i in perm[b0:b0+B]]
        if len(idx) < 2:
            continue
        LK = np.full((len(idx), 100), -9, dtype=np.int32)
        for bi, i in enumerate(idx):
            lo, hi = table["lab_ptr"][i], table["lab_ptr"][i+1]
            LK[bi, table["lab_cell"][lo:hi]] = table["lab_chain"][lo:hi]
        batch = (jnp.array(table["tiles_own"][idx]),
                 jnp.array(table["scalars"][idx]),
                 jnp.array(LK),
                 jnp.array(edge_lo[idx]), jnp.array(edge_hi[idx]),
                 jnp.array(edge_chain[idx]))

        def loss_m1w(p, *batch):
            l = loss_m1(p, *batch)
            return jnp.where(jnp.isfinite(l), l, 0.0)

        def step1(p, s, batch):
            l, g = jax.value_and_grad(loss_m1w)(p, *batch)
            g = jax.lax.cond(jnp.isfinite(l), lambda gg: gg, lambda gg:
                             jax.tree_util.tree_map(jnp.zeros_like, gg), g)
            upd, s2 = opt1.update(g, s, p)
            return optax.apply_updates(p, upd), s2, l
        params, s1, l1 = jax.jit(step1)(params, s1, batch)
        l1f = float(l1)
        if not np.isfinite(l1f):
            skipped += 1
            params = params  # optax state advanced; weights unchanged for NaN batch
        else:
            tot1 += l1f

        b2 = (jnp.array(table["tiles_own"][idx]),
              jnp.array(table["scalars"][idx]),
              jnp.array(table["chain_bag"][idx]),
              jnp.array(table["ord_kind"][idx]),
              jnp.array(table["ord_item"][idx]),
              jnp.array(table["ord_qty"][idx]))

        def loss_m2w(p, *b2):
            l = loss_m2(p, *b2)
            return jnp.where(jnp.isfinite(l), l, 0.0)

        def step2(p, s, b2):
            l, g = jax.value_and_grad(loss_m2w)(p, *b2)
            g = jax.lax.cond(jnp.isfinite(l), lambda gg: gg, lambda gg:
                             jax.tree_util.tree_map(jnp.zeros_like, gg), g)
            upd, s2b = opt2.update(g, s, p)
            return optax.apply_updates(p, upd), s2b, l
        p2, s2, l2 = jax.jit(step2)(p2, s2, b2)
        l2f = float(l2)
        if not np.isfinite(l2f):
            skipped += 1
        else:
            tot2 += l2f
        cnt += 1

    # held-out metrics
    va_sub = va_idx[:512]
    va_LK = pad_labels(va_sub)
    va_batch = (jnp.array(table["tiles_own"][va_sub]),
                jnp.array(table["scalars"][va_sub]),
                jnp.array(va_LK),
                jnp.array(edge_lo[va_sub]), jnp.array(edge_hi[va_sub]),
                jnp.array(edge_chain[va_sub]))
    v1 = float(loss_m1(params, *va_batch))
    acc1 = top1_acc_m1(params, va_sub)
    b2v = (jnp.array(table["tiles_own"][va_sub]),
           jnp.array(table["scalars"][va_sub]),
           jnp.array(table["chain_bag"][va_sub]),
           jnp.array(table["ord_kind"][va_sub]),
           jnp.array(table["ord_item"][va_sub]),
           jnp.array(table["ord_qty"][va_sub]))
    v2 = float(loss_m2(p2, *b2v))
    klog = jax.nn.log_softmax((jax_jit_hidden(p2, *b2v[:3]) @ p2["w_kind"]).reshape(-1, 24, 10, NV+1), -1)
    pick = jnp.argmax(klog, -1)
    km = b2v[3] > 0
    acc_kind = float(((pick == b2v[3]) & km).sum()) / max(int(km.sum()), 1)
    history.append((epoch, tot1/max(cnt,1), v1, acc1, tot2/max(cnt,1), v2, acc_kind, skipped))
    print(f"epoch {epoch}: m1 tr={tot1/max(cnt,1):.3f} val={v1:.3f} top1={acc1*100:.1f}% "
          f"| m2 tr={tot2/max(cnt,1):.3f} val={v2:.3f} kind_acc={acc_kind*100:.1f}% "
          f"skip={skipped} ({time.time()-t0:.0f}s)", flush=True)
    np.savez_compressed(f"{WORK}/m1_ep{epoch}.npz",
                        **{kk: np.asarray(v) for kk, v in params.items()})
    np.savez_compressed(f"{WORK}/m2_ep{epoch}.npz",
                        **{kk: np.asarray(v) for kk, v in p2.items()})
    _save_ckpt(epoch)
print("training complete")

In [ ]:
# 6. verify NumPy forward == JAX hidden states, then final export
# (the 64-dim chain_emb vs 704-dim tile_emb indexing is the one trap: keep them apart)
def np_hidden_m2(w, tiles, scal, bag):
    t = w["tile_emb"][tiles]
    pool = t.mean(axis=1)
    cb = w["chain_emb"][np.clip(bag.argmax(-1), 0, w["chain_emb"].shape[0] - 1)]
    ctx = np.tanh(scal @ w["w_sc"] + w["b_sc"])
    h = np.concatenate([pool, cb, ctx], -1)
    h = np.tanh(h @ w["w_h1"] + w["b_h1"])
    h = np.tanh(h @ w["w_h2"] + w["b_h2"])
    return h

import jax.numpy as jnp
w_np = {kk: np.asarray(v) for kk, v in p2.items()}
h_np = np_hidden_m2(w_np, table["tiles_own"][:4], table["scalars"][:4], table["chain_bag"][:4])
h_jax = np.asarray(jax_jit_hidden(p2, table["tiles_own"][:4], table["scalars"][:4],
                                  table["chain_bag"][:4]))
diff = np.abs(h_np - h_jax).max()
print(f"numpy vs jax hidden-state max diff: {diff:.2e}")
assert diff < 1e-4, "NumPy forward drifted from JAX"
np.savez_compressed(f"{WORK}/m1_final.npz", **{kk: np.asarray(v) for kk, v in params.items()})
np.savez_compressed(f"{WORK}/m2_final.npz", **w_np)
print("final exports written: m1_final.npz / m2_final.npz")